# jepa-as-a-tool: train the world model (v1)

Runs the whole GPU part: data → V-JEPA 2 latents → translator → predictor.
At the end you download one small folder, `runs/wm`, and send it back.

**Where to run it:** any notebook with a GPU (≥ 15 GB) and internet.
- **Kaggle**: Accelerator *GPU T4 x1* (or P100), Internet *on*, Persistence *Files only*.
- **Colab**: Runtime → *T4 GPU*. Files in `/content` vanish when the runtime stops.
- **Elsewhere**: files go to the folder the notebook starts in.

Only one GPU is used: a second one would sit idle.

In [ ]:
import os

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    WORK = os.getcwd()
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .

If pip printed that it upgraded **numpy** or **torch**, restart the kernel, then run the cell above again. pip's complaints about other packages (gradio, bigframes, ...) are harmless.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git log --oneline -1

## 1. Data (CPU, ~2 min)

4,000 episodes of 6 random pushes in the real PushT physics = 28,000 poses.

In [ ]:
!jat gen data/random-4k --episodes 4000 --workers 4

## 2. V-JEPA 2 latents (GPU, ~10–20 min)

Renders every pose and encodes it with V-JEPA 2 ViT-L (1.3 GB download).
Writes `data/random-4k/latents.npy`, ~3.7 GB (64 tokens × 1024 per frame).

In [ ]:
!jat encode data/random-4k

## 3. Translator = the probe (GPU, ~5 min)

Learns latent → pose. **This answers: does V-JEPA 2 see where the T is?**

- Good: under ~5 px and ~3°. Go on.
- Bad: over ~15 px or ~10°. Stop and send me the output.

In [ ]:
!jat train translator data/random-4k --out runs/wm

## 4. Predictor = the world model (GPU, ~20–40 min)

Learns (latent, push) → next latent, prints the time left, keeps the best checkpoint, and ends with
the table: imagined pose vs true pose after 0, 1, 2, ... pushes, next to the "nothing moves" baseline.

In [ ]:
!jat train predictor data/random-4k --out runs/wm

## 5. Send back

Download `wm.zip` and send it, with the output of steps 3 and 4.

In [ ]:
!cd runs && zip -r {WORK}/wm.zip wm
!ls -lh {WORK}/wm.zip